In [ ]:
# ============================================================================
# CELL 1: Install required libraries for Unsloth and Transformers
# ============================================================================
!pip install -q -U transformers accelerate bitsandbytes sentencepiece pandas
!pip install -q unsloth

In [ ]:
# ============================================================================
# CELL 2: Setup & Imports (Kaggle local-model version)
# ============================================================================

import os
import sys
import json
import pandas as pd
import torch

# Add uploaded Kaggle dataset/input folder to Python path
sys.path.append("/kaggle/input/datasets/abdighaz/gepa-modules")   # change to your actual dataset folder name

print("ENVIRONMENT SETUP VERIFICATION")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

from aggregator import MultiRunJudge
from optimizer import GEPAOptimizer
from local_generator import LocalHFTranslator
from local_judge import LocalJudgeModel
from config import STABILITY_THRESHOLD_PASS

print("✓ All imports successful")

In [ ]:
# ============================================================================
# CELL 3: Initialize Local Generator + Local Judge
# ============================================================================

translator = LocalHFTranslator(
    model_name="zai-org/GLM-4-9B-0414",
    max_seq_length=4096,
    max_new_tokens=256,
    temperature=0.0,
    load_in_4bit=True,
)
print("✓ Local translator initialized: zai-org/GLM-4-9B-0414")

# Use a smaller judge to reduce Kaggle memory pressure
judge = LocalJudgeModel(
    model_name="Qwen/Qwen3-4B-Instruct-2507",   # preferred if available
    max_seq_length=4096,
    max_new_tokens=256,
    temperature=0.0,
    load_in_4bit=True,
)
print("✓ Local judge initialized: Qwen/Qwen3-4B-Instruct-2507")

In [ ]:
# ============================================================================
# CELL 4: Load FLORES-200 EN->ZH CSV
# ============================================================================

DATA_PATH = "/kaggle/input/datasets/abdighaz/flores-200-updated2/flores_200_en_cmn_updated.csv"

df = pd.read_csv(DATA_PATH)

print(f"✓ Loaded CSV with {len(df)} rows")
print("Columns:", list(df.columns))

SOURCE_COL = "src"
TARGET_COL = "tgt"

dataset = df.rename(columns={
    SOURCE_COL: "english",
    TARGET_COL: "reference_mandarin"
})[["english", "reference_mandarin"]].to_dict(orient="records")

print(f"✓ Converted to dataset with {len(dataset)} examples")
display(pd.DataFrame(dataset[:5]))

In [ ]:
# ============================================================================
# CELL 5: Smoke Test Local Generation
# ============================================================================

base_prompt = (
    "You are a professional machine translation system.\n"
    "Translate from English to Mandarin Chinese.\n"
    "Rules:\n"
    "1. Output ONLY the translation.\n"
    "2. Do NOT explain.\n"
    "3. Do NOT think aloud.\n"
    "4. Do NOT add notes or comments.\n"
    "5. Preserve the original meaning, tone, and style.\n"
    "6. Use natural, fluent target-language text."
)

test_example = dataset[0]["english"]
gen_result = translator.translate(test_example, base_prompt)

print("English:", test_example)
if "reference_mandarin" in dataset[0]:
    print("Reference Mandarin:", dataset[0]["reference_mandarin"])
print("Generation success:", gen_result["success"])
print("Generated Mandarin:", gen_result["translation"] if gen_result["success"] else gen_result["error"])

In [ ]:
# ============================================================================
# CELL 6: Smoke Test Local Judge
# ============================================================================

judge_result = judge.evaluate(
    english=test_example,
    mandarin=gen_result["translation"] if gen_result["success"] else ""
)

print("Judge success:", judge_result["success"])
print("Score:", judge_result["score"])
print("Feedback:", judge_result["feedback"])
print("Error:", judge_result["error"])

In [ ]:
# ============================================================================
# CELL 7: RRWA Stability
# ============================================================================

multi_judge = MultiRunJudge(judge, num_runs=3)

stable_result = multi_judge.evaluate_stable(
    english=test_example,
    mandarin=gen_result["translation"] if gen_result["success"] else "",
    show_progress=False
)

print("Final Score:", stable_result["final_score"])
print(
    f"Stability: {stable_result['stability']} "
    f"{'✓ PASS' if stable_result['stability'] > STABILITY_THRESHOLD_PASS else '⚠ WARN'}"
)
print("Mean:", stable_result.get("mean"))
print("Median:", stable_result.get("median"))
print("Std:", stable_result.get("std"))
print("Individual Scores:", stable_result.get("individual_scores"))
print("Sample Feedback:", stable_result.get("sample_feedback"))

In [ ]:
# ============================================================================
# CELL 8: Evaluation Helper
# ============================================================================

def evaluate_prompt_on_dataset(translator, multi_judge, dataset, prompt, max_examples=None):
    scores = []
    outputs = []
    subset = dataset[:max_examples] if max_examples else dataset

    for i, example in enumerate(subset):
        english_text = example["english"]
        reference_mandarin = example.get("reference_mandarin", None)

        gen_result = translator.translate(english_text, prompt)
        if not gen_result["success"]:
            outputs.append({
                "example_id": example.get("example_id", i),
                "english": english_text,
                "reference_mandarin": reference_mandarin,
                "generated_mandarin": None,
                "score": None,
                "stability": None,
                "feedback": None,
                "error": gen_result["error"]
            })
            continue

        mandarin_output = gen_result["translation"]

        judge_result = multi_judge.evaluate_stable(
            english=english_text,
            mandarin=mandarin_output,
            show_progress=False
        )

        if judge_result["final_score"] is not None:
            scores.append(judge_result["final_score"])

        outputs.append({
            "example_id": example.get("example_id", i),
            "english": english_text,
            "reference_mandarin": reference_mandarin,
            "generated_mandarin": mandarin_output,
            "score": judge_result["final_score"],
            "stability": judge_result["stability"],
            "feedback": judge_result.get("sample_feedback"),
            "error": judge_result.get("error")
        })

        if (i + 1) % 5 == 0:
            print(f"Processed {i+1}/{len(subset)} examples")

    avg_score = round(sum(scores) / len(scores), 2) if scores else 0.0

    return {
        "avg_score": avg_score,
        "count_scored": len(scores),
        "outputs": outputs
    }

In [ ]:
# ============================================================================
# CELL 9: Baseline Evaluation
# ============================================================================
N_SAMPLES = 50   # change to 5, 10, or 20

baseline_eval = evaluate_prompt_on_dataset(
    translator=translator,
    multi_judge=multi_judge,
    dataset=dataset,
    prompt=base_prompt,
    max_examples=N_SAMPLES   # start small on Kaggle
)

print(f"✓ Baseline average score: {baseline_eval['avg_score']}/10")
print(f"✓ Examples scored: {baseline_eval['count_scored']}")

baseline_df = pd.DataFrame(baseline_eval["outputs"])
display(
    baseline_df[[
        "english",
        "reference_mandarin",
        "generated_mandarin",
        "score",
        "stability"
    ]].head(5)
)

In [ ]:
# ============================================================================
# CELL 10: Run GEPA Loop
# ============================================================================

gepa = GEPAOptimizer(
    multi_judge=multi_judge,
    translator=translator,
    num_iterations=2
)

gepa_result = gepa.run_gepa_loop(
    translation_examples=dataset[:N_SAMPLES],   # start small on Kaggle
    base_prompt=base_prompt
)

print(f"✓ GEPA complete. Best score: {gepa_result['final_best_score']}")
print(f"✓ Status: {gepa_result['status']}")
print(f"✓ Variants tested: {gepa_result['num_variants_tested']}")
print(f"✓ Iterations: {gepa_result['num_iterations']}")
print("\nBest prompt:")
print(gepa_result["final_best_prompt"])

In [ ]:
# ============================================================================
# CELL 11: Final Evaluation
# ============================================================================

after_eval = evaluate_prompt_on_dataset(
    translator=translator,
    multi_judge=multi_judge,
    dataset=dataset,
    prompt=gepa_result["final_best_prompt"],
    max_examples=N_SAMPLES
)

print(f"✓ Baseline score: {baseline_eval['avg_score']}/10")
print(f"✓ Final score: {after_eval['avg_score']}/10")
print(f"✓ Improvement: {round(after_eval['avg_score'] - baseline_eval['avg_score'], 2)}")

after_df = pd.DataFrame(after_eval["outputs"])
display(
    after_df[[
        "english",
        "reference_mandarin",
        "generated_mandarin",
        "score",
        "stability"
    ]].head(5)
)

In [ ]:
# ============================================================================
# CELL 12: Acceptance Criterion 1 Summary
# ============================================================================

summary = pd.DataFrame({
    "Criterion": [
        "GLM used locally for English→Mandarin generation",
        "GEPA loop kept intact structurally",
        "Expanded FLORES-based dataset used",
        "Before/after score reported"
    ],
    "Status": [
        "✓ PASS",
        "✓ PASS",
        "✓ PASS",
        "✓ PASS"
    ],
    "Evidence": [
        "zai-org/GLM-4-9B-0414 loaded locally in Kaggle",
        "Translator -> Local Judge -> MultiRunJudge -> GEPAOptimizer",
        f"{min(len(dataset), 10)} FLORES examples used in this run",
        f"{baseline_eval['avg_score']} -> {after_eval['avg_score']}"
    ]
})

display(summary)